# Disney Cruise Line scraper — Challenge 1
Scrapes `disneycruise.disney.go.com/en-in`, cleans the data, writes `results.csv` and prints the 5 answers.

**Run all cells top to bottom.** The site's bot protection (Akamai) blocks Colab's datacenter IPs, so the live scrape (cell 4) normally ends in *Access Denied* here. The notebook then falls back to the `raw.json` that the same script scraped from a home connection (35 pages, 171 cards), and runs the identical cleaning + answer code on it.

## 1. Install

In [ ]:
!pip -q install playwright pandas
!playwright install chromium --with-deps > /dev/null 2>&1
print('ready')

## 2. Scraper source (`disney_cruise_scraper.py`)
Steps: navigate → View Dates → wait for cards → scroll 35 pages → clean → CSV → answers.

In [ ]:
%%writefile disney_cruise_scraper.py
"""
Disney Cruise Line scraper  (https://disneycruise.disney.go.com/en-in/)

Steps (mirrors the challenge brief):
  1. Open the homepage and accept the consent banner.
  2. Click "View Dates" to open the results list ("Find a Cruise").
  3. Wait for the cards to load.
  4. Scroll to the end, 5 cards per "page", until all 35 pages are loaded.
  5. Clean the data (no empties, no duplicates, locations present).
  6. Write results.csv and print the answers to the 5 questions.

NOTE: the site sits behind Akamai and returns "Access Denied" to datacenter IPs
(e.g. Google Colab). Run this from a normal home/office connection.

Usage:  python disney_cruise_scraper.py            # scrape + clean + answers
        python disney_cruise_scraper.py --clean    # skip scraping, reuse data/raw.json
"""
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd

URL = "https://disneycruise.disney.go.com/en-in/"
CARD = "dcl-product-card"
CARDS_PER_PAGE = 5
MIN_PAGES = 35
# Local layout: src/ (this file) + data/ (outputs). In Colab everything sits in the cwd.
_DATA = Path(__file__).resolve().parent.parent / "data"
DATA_DIR = _DATA if _DATA.is_dir() else Path(".")
RAW_JSON = DATA_DIR / "raw.json"
API_JSON = DATA_DIR / "raw_api.json"
OUT_CSV = DATA_DIR / "results.csv"

# Runs inside the page: turns every card into a plain dict of raw text.
EXTRACT_JS = """
cards => cards.map(c => {
  const t = s => { const e = c.querySelector(s); return e ? e.innerText.trim() : ''; };
  const all = s => [...c.querySelectorAll(s)].map(e => e.innerText.trim()).filter(Boolean);
  const btn = c.querySelector('a.product-card-footer-wrapper__btn');
  return {
    title: t('h2.product-card-content__name'),
    banner: t('.product-card__banner'),
    locations: all('ul.ul-port-list li'),
    prices: all('.wrapper-price__pricing'),
    price_text: t('.product-card-pricing-wrapper'),
    dates_button: btn ? btn.innerText.trim() : '',
    link: btn ? btn.href : '',
    full_text: c.innerText
  };
})
"""


# --------------------------------------------------------------------------
# Scraping
# --------------------------------------------------------------------------
def scrape():
    from playwright.sync_api import sync_playwright

    api_products = {}

    with sync_playwright() as p:
        # A real Edge window is far less likely to be flagged than headless Chromium.
        if "google.colab" in sys.modules:   # Colab: headless Chromium (usually blocked by the site)
            browser = p.chromium.launch(headless=True, args=["--no-sandbox"])
        else:
            browser = p.chromium.launch(channel="msedge", headless=False)
        ctx = browser.new_context(viewport={"width": 1440, "height": 900}, locale="en-IN")
        page = ctx.new_page()

        # Keep the JSON the site itself fetches for each page (used to cross-check).
        total_info = {}

        def on_response(r):
            if "available-products" in r.url and "json" in (r.headers.get("content-type") or ""):
                try:
                    j = r.json()
                except Exception:
                    return
                total_info.update({k: j.get(k) for k in ("totalPages", "totalAvailableCruises")})
                for prod in j.get("products", []):
                    api_products[prod["productId"]] = {
                        "productId": prod["productId"],
                        "name": prod.get("productDisplayName"),
                        "sailings": sum(i.get("numberOfSailings", 0) for i in prod.get("itineraries", [])),
                    }

        page.on("response", on_response)

        # Step 1: navigate and agree
        page.goto(URL, wait_until="domcontentloaded")
        page.wait_for_timeout(5000)
        if "denied" in page.title().lower():
            raise RuntimeError("Access Denied by the site's bot protection (datacenter IP?). "
                               "Run from a home/office connection or use --clean with data/raw.json.")
        for label in ("Accept All Cookies", "Accept All", "Agree", "I Agree", "Accept"):
            try:
                page.get_by_role("button", name=label).first.click(timeout=1500)
                break
            except Exception:
                pass

        # Step 2: View Dates (no filters = the full catalogue)
        page.locator(".view-cruises-button").first.click()

        # Step 3: wait for the first cards
        page.wait_for_selector(CARD, timeout=60000)
        page.wait_for_timeout(3000)

        # Step 4: scroll until every page is loaded
        total = total_info.get("totalAvailableCruises") or MIN_PAGES * CARDS_PER_PAGE
        count, stalled = page.locator(CARD).count(), 0
        while count < total and stalled < 8:
            page.mouse.wheel(0, 8000)
            page.keyboard.press("End")
            page.wait_for_timeout(1500)
            new = page.locator(CARD).count()
            stalled = stalled + 1 if new == count else 0
            count = new
            print(f"  loaded {count}/{total} cards (page {count // CARDS_PER_PAGE})", end="\r")
        print()

        rows = page.eval_on_selector_all(CARD, EXTRACT_JS)
        browser.close()

    for i, r in enumerate(rows):
        r["page"] = i // CARDS_PER_PAGE + 1

    with open(RAW_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=1)
    with open(API_JSON, "w", encoding="utf-8") as f:
        json.dump({"info": total_info, "products": list(api_products.values())}, f, indent=1)
    print(f"scraped {len(rows)} cards; site reports {total_info}")
    return rows


# --------------------------------------------------------------------------
# Cleaning
# --------------------------------------------------------------------------
def money(s):
    """'₹1,09,674 INR' / '$1,142 USD' -> int"""
    m = re.search(r"[\d,]+", s or "")
    return int(m.group().replace(",", "")) if m else None


def split_port(text):
    """'San Diego ending in Galveston' -> ('San Diego', 'ending in Galveston')"""
    m = re.match(r"(.+?)\s+((?:ending|with)\s.*)$", text.strip(), flags=re.I)
    return (m.group(1).strip(), m.group(2).strip()) if m else (text.strip(), "No Note")


def clean(rows):
    recs = []
    for r in rows:
        title = re.sub(r"\s+", " ", r["title"]).strip()
        nights = re.match(r"(\d+)-Night", title)
        port = re.search(r"\bfrom (.+)$", title)
        locs = [re.sub(r"\s+", " ", l).strip() for l in r["locations"] if l.strip()]
        prices = r["prices"]
        inr = next((money(p) for p in prices if "INR" in p), None)
        if inr is None:
            inr = money(re.search(r"₹[\d,]+", r["price_text"]).group()) if re.search(r"₹[\d,]+", r["price_text"]) else None
        n_dates = re.search(r"Show (\d+) Dates?", r["dates_button"] or r["full_text"])
        offer = "Guaranteed with Restrictions" if "Guaranteed with Restrictions" in r["full_text"] else ""
        recs.append({
            "page": r["page"],
            "cruise_name": title,
            "nights": int(nights.group(1)) if nights else None,
            "departure_port": split_port(port.group(1))[0] if port else None,
            "itinerary_note": split_port(port.group(1))[1] if port else "No Note",
            "locations": " | ".join(locs),
            "banner_tag": re.sub(r"\s+", " ", r["banner"]).strip(),
            "offer": offer,
            "price_from_inr": inr,
            "num_dates": int(n_dates.group(1)) if n_dates else None,
            "link": r["link"],
        })
    df = pd.DataFrame(recs)

    n0 = len(df)
    # Duplicates: the same cruise card appearing twice.
    df = df.drop_duplicates(subset=["cruise_name", "locations", "price_from_inr", "num_dates", "banner_tag"])
    print(f"duplicates removed: {n0 - len(df)}")

    # Empty-field rules. Locations must exist; the cards without a "Sailing to"
    # block fall back to the destination named in the cruise title.
    df["locations"] = df["locations"].where(df["locations"] != "", df["departure_port"])
    df["banner_tag"] = df["banner_tag"].replace("", "No Banner")
    df["offer"] = df["offer"].replace("", "No Offer")
    required = ["cruise_name", "departure_port", "locations", "price_from_inr", "num_dates", "nights"]
    bad = df[df[required].isna().any(axis=1) | (df[required].astype(str) == "").any(axis=1)]
    if len(bad):
        print(f"dropping {len(bad)} rows with empty required fields")
    df = df.drop(bad.index).reset_index(drop=True)
    return df


# --------------------------------------------------------------------------
# Answers
# --------------------------------------------------------------------------
def answers(df, api_total=None):
    pacific = df["cruise_name"].str.contains("Pacific", case=False) | df["locations"].str.contains("Pacific", case=False)
    holiday_tags = ["Very Merrytime", "Halloween on the High Seas"]   # site lists these under "Holiday Cruises"
    holiday = df["banner_tag"].isin(holiday_tags)
    themed = df["banner_tag"].isin(["Marvel Days at Sea", "Pixar Days at Sea"])
    two_plus = df["num_dates"] > 2
    miami = df["departure_port"].str.contains("Miami", case=False).sum()
    london = df["departure_port"].str.contains("London", case=False).sum()
    print("\n=========== ANSWERS ===========")
    print(f"(i)   Pacific cruises .............. {int(pacific.sum())}   ({int(df.loc[pacific, 'num_dates'].sum())} sailings)")
    print(f"(ii)  Total cruises ................ {int(df['num_dates'].sum())}"
          f"   (= site header; {len(df)} distinct cruise cards on {df['page'].max()} pages)")
    print(f"(iii) Holiday cruises .............. {int(holiday.sum())}   {df.loc[holiday, 'banner_tag'].value_counts().to_dict()}"
          f"   [+{int(themed.sum())} themed Marvel/Pixar]")
    print(f"(iv)  Cruises with > 2 dates ....... {int(two_plus.sum())}")
    print(f"(v)   Departing Miami / London ..... Miami={miami}, London={london}, total={miami + london}")
    print("departure ports:", df["departure_port"].value_counts().to_dict())


if __name__ == "__main__":
    if "--clean" in sys.argv:
        raw = json.load(open(RAW_JSON, encoding="utf-8"))
    else:
        t0 = time.time()
        raw = scrape()
        print(f"scrape took {time.time() - t0:.0f}s")
    frame = clean(raw)
    frame.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"wrote {OUT_CSV}: {frame.shape[0]} rows x {frame.shape[1]} cols")
    try:
        api_total = json.load(open(API_JSON))["info"].get("totalAvailableCruises")
    except Exception:
        api_total = None
    answers(frame, api_total)


## 3. Data scraped from a home connection (fallback)
The raw cards, exactly as the script's `scrape()` saved them.

In [ ]:
import base64, gzip
RAW = 'H4sIAMvDwGoC/+1dzY7cRpK+DzDvQHiBGXmR4+H/z1wWkmzJsiVZUHtl7IwGA3Y11U13NdlgsSRrBnvZN9nTnhd72rPfZJ9kk8lkdSWLVZmRDLLI7gJkwB1dUU1GfF/+REZE/uW3vzH+Qf8zvijTcpl88SfjC+cPr9PLq9J4WqzTVWJ8KPIb4yzNLuPbvEi+IOzD53GWJUX1aS5Y5ou4TPNsRWV/+Wstuy3SRcIE1Y/GF//3H/9rETMifuDWWqKokmxr/q1MfimrP/Gm+ok9xz+/z7Y0jBev377PbOP5OlmVq/fZ27hMjK+TMk6XK+N3xvMiLtdpmSb0V2dX+SfDd42v6UdW/JEvqv//2/m6LPOs+jNdn1mm2XX1u6uyvF396Y9/vEhXWfJ5wSzzVf3DV5f5V4v85o//xFU+rJfLzaM/X8dFnJVJcmF8Sssr4y19Uvo2zFLG++xr9g2NpV+mWUI/tlwaq2SZLErjc74ujFVJn6fIqRM+5EUlMuKVcRsXpZF/MMqrdGWsbpNFGi/pzx+S4ivjZRIXmfGKOut9dtCX77OfruLy9yvjRbZYri/oM+aZERvCQ/3L+2wg89/GlxXcLPrDv5NdFLqDotCxiReaAgq5CIDCWgNqBieUo7D1mbmj0B0EhSjmV0Thyzi7vI4/pT3gWEOt+SJivIqX8edVGu8gro1VNyCOJY6YXATAaq0BNZYdyLHa+kxfrKpZXBE09DXoq1Elo8yrn3at3wUsFFtJgNUMzE/in2PxFeOMvkZymQNA9U22SrL4IqavlfySLnIppKyQmKYlQIqLAJCqNcBmshQgZd3LSXi/rzXh3PZ7F5hRvKQ4Sh4HzLZLXC8QwMxFADDXGlAzRXIsR/dyJj8KlDF8hD8sv0uKz/Shi+Jzmd4kmLgOSWiJa1QuguCaaUBtprBRwt0niUZ8nz3oIRvDZxzn9hjLj6dxGdNXjI0Xq2WcXRDjKf2RGj2ji1uiCX6HzlyuLW7QahFkg8Y0wDsEhf3ZaX2iAvYDwFDhAYb7JDxwJ8+DiJi2uFLnIggPmAbUkJZCoMIKT8ubMZiA4UDlKeEqvknp+26//7O8qLbk9IWLi3gJCXhwy7zM8+t8XdL3/mzE9KuqP3WVr+n3v8nTrJQHQUxiuo4YBKlFkCAI0zjNB8rzgRwJmnxQQ0VncAbBh/iTwmCbANcjgS1uArgIgnumAR4+mM0Ojv7bn7hvW4CjbnURPDbkcK8Kd8XBn6P7dbxaxWtSP1K8UgqMh/5OYJyKgIHx0J984Ge6G+QjzxEt0OyL5/d1MWeTM/6UMcSuwg3pv9ZqqhZBmMM0wKfACmfw95o5U91jYLgTnyXfxstl/ilJssoG5VVifEu/wThL4hXmIiskQeS26VCJYHSgGhNfZO0350NfcPX3ngT8zkTB7znEi0IB/FwEAH+tATWfQoAJN740Zfgf9cgBw3+Kg/+beJF+SBfG0zxeld0vaiTZRfX8aWa8i7NFvv5I+aAectroGI+eFPR5V1f0jy3XN+dp/GU1K1bGkNMiIpYfibSoRRBaMI3JB5x+yJI//BQ3ELobiTVdpYlTFbd1YhfByMpj9zFio5rbY2oXr5Vpy0UwBHvwVE9bYZVv4y7z57QhRTEqh6yrPdze8e2OwzrHXhsdYWsqQ6fvkrCVtcNFAHTWGmBDKoDzyOPrQd9owrfTT13wxLCqIjp/ot+d0CXQ07hIz8+TFm2fx8uP9APUL7rBkzdFflkkq1x5Nex7xHVbsKxFEFgyDagBPTksvQcRGVEEhSYN2pDoZACCAyUM8PjLsoegr/g2/ZgmxZA7w6fxec6+8uV6Ea82jNCNIfo+8VtBEy6CMIVpTD49bcr7RhCQtEOLXdBR2VhiOFjCpIMrvzfVyq/aO9C5NF4CVjXt1TYRluxP6YQdf6qm9PiznCkhsdwWU2oRhClMAzwkKSzEPf/hZC+IcNBkQwsaHZDoZAKGAxWXVVr7VJ055TBL+ib9+BGxIzEJlIsgxGEapykGJTI/2N66m1Ma+UEY7uYs8wbavCBuVwKX2Ja4XeEiAENqDfDZh0KY0nIGqQ08/gYBxWSKMBttMIeFI3VWQtRsYavsmotgcA3hdb+jB30e+ICuBibFtROGyyVsc6bNNmjwPwiI44tLJy6CMI1pzHrpNKdjAQxzDzmnDJP1qTWPBCTy3Da6KxEM3VRj8jkL081fm8ncgeDmIVl1rBkiNInpiTmgXATgUK0BnphVFmP2Pa0sm/48hOJUThl/kOMP9dxp1COP0CKWKRYccBGEMkxj8tuXqU070z7mwHCqhDF9jjlU+YJ46BFWpXpOmyuVCMYVqjGDyNSsynOOeAiC4lAJUYZdjQkkwD3uoMYJoqBNmUoEowzVAFtYIfnE8k4rMq0V2TZk9A82UByrTJ2fh81EwS9pC52dYlAugpDH0akUHD1x61DAa6rlZRimHXCFhH/4DVstBcQPxIMRLoKgl2mARwiFrYVlP8AynImtnDCcywkU7CGQP8qmfGdoJwc36zLqRFXxnnjSwUUA6tQapw4ZIKr4I2zJ22DpBkkXYRBcKqGLO/J8M/wOhNosdM02lyoRjEtUYwY7kNme0KNMQ6ibExSfS+jmjVQz8jT/+/omWbanJ3BuVuQSy/REMtUiCJmYxqk9jWa0eNhMsRZSVBLHMPw56CpuApUl1EaRa7WJU4lgxKEap7Tfia/v9I9cMBwsYdJUc8KkBIqI22oayEUQAjGNGfezmXQbSwTrcvCGo5ZFwYNbNjFNYlrbodmNSBmOjcbku405x1i37wIMx14ShLnTLrxjNvCFGxQ2IhjwfHg3fkcBeY7/cA7UjhY5RXKgIhV096av4uJjsqR/8fOqmkHoBIZWJEQNYBNP6J29EUFowDQm38po15DT6SuAY8Q+g/Ig2Q+wxFO7uj0yNFtwrEUQODIN8PpJYVS2/EFq1sZNM5BmeSJZcMAVArDaYLCouE0si5ieJwCWiwCArTXA5lYISFjuw603mFokHMnPnFbREWmFuAy3bBKZocifWgThD9M4XWR2TxblGO6UsMSb9t1O1AYOcV0xLsJFEF4wjcnzwptkWhqO+fqM1uM3ZYEN3Q4J3agN0UoEgyjVOHVGn1//Feko3t+z+IP48S7WoAYJiCnc1b4RQfjCNGZXITbZIR7BnIooxe8IhJOMQo1Q3SDYWoPXIggwmcYM9rATTf9AsmBzh7U5oXOZ3jEWapj2QqMWAfGpMR2Nn3l4rF2ZTjwDw6YyxParBh8pMYnZwt0dRF3wIOrCh4DRZ/cp11vjGFEVlGOc6WFN87ZFLKEf4EYEQGitMfkquK6jvslO/Bg2VQXs7Ko3qXkcYnrivomLILhlGjOOM0x1B4Vg2CHXrEeOkdkecRzxPJuLINhlGqemlvfwqAPDtQ19rCM2h9HlRmjtcCO0oNwIrcmnHj28HjAAAvT1n4wA3lR6ctPXDYlptzBfiyCYZxoTX8t4k0msQzCXKsLmU+2rmZ/nuMRtpYtyEQC/tcbk6xbncLHPBBOfFNILMQAwJCVRUl1xY+BHJN4DLFc4Cr3QwvIobu61q8DNJZdywydhJC6quAjCDaYx8UWVc5SE8U6M9TdXgzD7mP3oEWM9TkSisDVE1yIIDJnGqdnQ/ciBQvDmJFgy7KbCNUkQiDUUXAQgTq1xygG/t73ncVwso5Na2GKQWrn2Oa+UNhb9J843XAShDdOYQa3cPLoGDbYGa4Gjkx4YrpTxIxi9gRCwJWSTHbFOijI33tE/SB2sftjs2iQQrhXdiCCkYhqnA7sedAsm0z9yD5I6GYjg93EmKK3cju48JJ2tkusQxxarm7gIwjKmceriNZNJrQUfxR0UhpMbSjmj5kQPfdMdtY1LfL81V9UiCIuYBjhYaCnEhK1TC5tjXW2H5FdV5nwTA8YPI8kuqpdOMzbZfreOs1GvWuGfbv42MfgM/1ZpfRgSO2jNXLUIwjmmMfEg3w9Z8oef4gaud/jv6+uxzyM7/dzJmP5ekfHFn0ctg2cSxxFBzkUAkNcak98E+ZOuZcAwIs4g3tBoi9FHXBo1u3zhCYjxbJkX6UUsB7i103WEiyAAt3TaVIwOcL1xXOrusdc4e1zdSRoEx8hIo7ZzGr6NFTRo7AXUNuKegYsgyGcak++3Os/G80OllyhEkDH82hDHHSAva7y2b5onmV5IXF88kuEiCLuYxukk80GlRGI4XZV6h2d+0VTN/L93pz46J8/Kr4wfr/KbaiH9r1+dfWW8S4tLujypy9dW/bb3XkSCUGw1wEUQAjONmeXwtJeJGGAZfat/GBrKkQAEB+JwsWMVjpXSo0gjtK1a67nV92q+SSJPnFO5CEDJWuNBUFKOGU1aShCjvbPbh4wuZiL4sdcCVT/xp195QI9TIt8hgSlOaVwE4Q/TON0c3m8VOmQqKs6pEYqfB2XYMCmriC3sA5O4jtiugosAfKs1TnvA6bFPm24Km0AMrzfk84YMXI4SqgwsEkbixMVFECIxjcn3K5p8aBDDjvrQVMHkm/SXuFDrqIUI0jAkli821eIiAEhrjYnvTnbseyigfWy4IlhUhlZn8pfpRCblrLhz5iIANmuNU12k3oyOYD0ZDoND767T1GT4LWkUEMsWB00uggCTaUz++F3NPVPIGMQwqCpYsROg9IpEXsV/j8vlr/+ZKU/3UUhMVzw+5yIIcpnGxIfU6RRg7HipE7z9bdpg158FdnULnKihLDNoI7gSwRBMNSafADLLIiIM28qgfPxLwvr1w3Gqi1sdazs4sBEpw7jRmNmp0Gz6w6PYV4ZklHAWcgs2WAzBqW5+NaNQxHItgmCZadyDBscTDYGhWFh1iQHcy0G6yaM1vGEGsZ0d3FYiGG6pxuS3cV0N5Qfd2uk0nMGxpgykzoSO76QQrbqTt5YJtQgCUaYxs2WCM5WjsE6Y9rdog9JAMpSCytUGi5J1ZyPyX/5IHyJf0k3Bk4K+7uqqnYolBblHgsAUQV6LICBnGpMfh6fSlwIEq0H51QktCaQ6OYkAAFVS6oVQVOeOMQMq1G4+cYXukxsRhH1MY3bXOs4wwIJjaxnQ+1S54BxkYy2gQmqs1gKqFkHQzTTme749wZKSTmD3N7MM1/7otcJvrqi9blfn6+KSGFUK8quYEjzJgOUgRy45ps6JSCTcvrYRQYjENGbWOMI/asHxPvwMlc2uXKeM4k1Vvg6TpNKfTUOWYzrEqiJ2vsA5LgJwrtaYOOf8CSe+ItVIonii4UvYa34bJLsLRgW+3n2zjKvF7tf5TZrVK/Dkdn2+TBdybrjEslrcqEUQbjCNyaep++Nv2uEEkHm0kxEI9h+EEtKy4ekUCksXmBqFxNQzPolscd/ERRB6MY17udyT4GVilcN7F5FqFcUonpTxdKSQA+D8EDcSYUXE9z2RUbUIwiimAfVDIJ+wAtQJ6yixCPmJo35oAsPuMgJsTryS5WW6vhF3l/R56VvdwjKh/5wk58X68jIxHj0p1qtVslx9SZrvlwLWtkjYAiwXAQBba8wl/f6w5TVxedALXWBDsJkq1l5T4l1Vs9w36yK/TRAw9zYvy2pOvDEePb5Z1f9LX/d1Uv0dtfPA6kruVsyWiyDAYxozAZ6iGzQBKHdJJwr7G7BBYSQ5Tnu8jFfXrSnhXZwt8jWdDwDYe76MF2lS0MVP8ql6/UdnZXpdnaG+WCSL+Ja+c/2XNkvj6/iSTg1t8YvFZ7pIKuK0rCeE9u+/T8rFVXpdLZv4L6SI9olt2yKiaxEE0UxjBledKLlVE8uqLm65dp9Ld1zZyQQMw8u40KupgE4C6NEWwnZIIqtFhloEIQPTmPy96SoJo0e99U1/SYzhAdUJYvTs6D13hTxP8oKu437MP9EBg75bdlFZ7CbeBFuexcubar4mxnfU4Oki7pOf6pjEjcQjNy4CEKXWmHypyxzu8hkhs7uFu71428GZYn4tBhpwSDvElAaLr+7lqpTl0AKJ6hpNWzyr4CIIkZnGjAsk5kHjMQ8bdxC4F3kK5R0I+BhlOp74pOuadGUjrk65CMDVWgOcoqcQp7WDe5mOfA+mVhSfcwba5hG7jw9b+FpdgutGbX5VIhi/qMbMKlq8+RS+ItgXEcn10GQbZ2V+yw6UdoMT457Ma80rFvFbUQ8uguCeaUw+X2WPq0AUOOj0sY/XVacABPf04g1W9SJika3rE9sVgxhcBME905jZeP9gW/lKadLfmzKWBGNebzSNfYtPfCdq86wSwXhGNU4XIz3gXQwCAlTJCbmaAyU+KCtU1irJgbI0Epprb0QwlkbwNsuj9wSeQywQAkFNxkpAp1bHo0Db3pBoaGuNe7HE8AfObkD/tUhXiyCkYxqTr3D2jtJJtkekAcGsMuC6413Z1xuqITWHeFLERRCoMo3Tbgk0I7hj1UrrkKS/Q5UH96S4WV+IHRBeJ5+Mf8uLa5WL73WWZd/TL6XWpEYtPpDmCTbJdod+qVfb4pnEdR2BZFwEIFmtMePj2H23cwERoEmGTrd2CpVrVBA8ImMJys6l//7kUDMmnV1JZTnf3yGE70MJ4U+/lvK08zh4naTifgPD2TKy+UfODsdPA/dcErbygLgIwjOmMZN6/mOmgPfK+UawcoNve6yLLxBPbbyAOKHYGpWLIEhlGpPP0Z7KjDChNqwYnlOFf79WFcNcp3hwhSVtXdGHd3SnZ4qhKi6C8I5pnPb/97Expgx8qgTvD5Fx+I3G6J7NNBSIG+wSNwATNzDd2VwJNSGq6LW/2MOM3j7AocbgYYThjjm9iJih2FOWiyBsYBpgTyicc9reKaQw4cNMFMfLONjE+89u4zRD6HvwJF2exzmpv09KEN8idiBmxXERgCC1xsTXeUp21kSjYPMuJCFYqMGRM+gyZ/hqtwPH7P12O1KoOyQIxTYyXASBOtM4Fb/do+1NByC1djydvO8PF1XeT6x3yXfrLKniPX1D1b5PPF88I+UiCGuZBtgNkULHkui+dywRHakUs0Yxtyruh70CY/dmW6U7MHZuzpDCPCC+11qH1SIIzJnGKd6mNSENefXGDoq60dNGTSe3+vsYZyWJlK+muPlXDCXoxAj8kLieeBjKRRDqMY3JJx3MM9iNEjGQAE0CMMWIAQYMxtnojXJCNWBkLzBJGInrQi4CsLbWmMkF8hM/EMIJuSF4pOGPO3/+aJ8CUTNGrtVmRyWCsYNqTD5tYi4HpoCzHwzLy2iAHXd+SZfMebH+9b9j1dhz4BHTFQNyXAQBKdOY/FHlkNHnHct3QgrBTuOMrMfIpNEdZ0OL2I44znIRAMK1xmnbfi/TZACjPgIOVBmqUpXXND5ZKXW7GYa1Bw6LEJNMQ484ZovFtQjCYqYxMxbvaWsz6UxUCEzHPjjSy3JFgI4q8dlqQ355JCTfoEgvLpOSNX54Ehfn8UW+uguSV7hJVvXQ93JN/d386m2+YscKzTnadsjgu/wq+z1VepyV6eVadkbkEtMkfrjdcmQjUuZvozGTWICiG3XTGTpc2uHKtgs7XLeLdhRDN2j3JBU7E+sPCp+bqLVsajBLxHYtgmCbacykVGdCzWuUxm8U88oQ3Xf8Vl2e/UDf/mf6/PEF5XCxPt+Myz/RCT65qX/xdF3Ev/5XnA8yltvEiaI23p0oAuLdiebWlnOcsX3HwV2O1R3Z+5tddWSfY7dOtB2Ly26ot0KRJbUIwhKmMXGW7G286c+0hafOFgHFV72ohRy0O0brW5fYJvFsV2ANFwFYU2uc9vkY+3x/wk2iFImJAAfVtd9MDvyhKz4nImZkCqzkIgAra41TxRreWb7Ceg/D6A34/dEDV3u3Ot/TJUPVX+jnazpL51mcFonKLkjzEMmtunXbdiAQgIsABKg1TuGrri1Oh0O7HKl+VoNibhnyJ9BgvV8XQbfKYwhaQzsXAZBda9ynOzaG7HkO7uqHYuB5jeL74lXaI3hQ1TyIQVouguCcaZxGcMURvO1EyOiNYGpVxA+1aN+L1WF681GbuTuBWS6CYNydQWB2zEX6Pjfq98RDMbIM3b7GAf3ZPc8jYYb3W/ElLoJRxL8v8SVYhGfAMO04mRwozmuoF+yhnuVuDU/n+Tq7MN7EGX1GFu9edtetbfXL1Wnod7jYkY9Q7+Llko5d8c6vt57uLjGEfvAyyeiHn+bL/OY8Vb/0dvMCxPgx+YUyVkpLn5imLdKyFkFoyTQmTkuxlfKxo7toSNU95e9E7R60bj9ZBzr3orKNxs5RoT92lEeFKntCYuvNI7e6aHdXSh8aFfaSmP786/9Uixf2DIdHAlAd9dbqlj0u/dPxMqXgzFJ5pDn0ieOK4QguAgwEtcasBoItKvaFhzYVd6AiQkSkn0pxdCcEutiH4DAZ+zQ76A6z1m0vaA+zwiOmRWyhknIjUmZFozHTjJtBGt5qZQPs4hfFtDL8NtHKin4XMf266p6Fb7LLJZvnOu5fQOkbRe2VleycyXhUffGTYp2tPqWL6y8Jf5IGlE/yeoZ9VVlrcUWtV5YKwI6ILdzwvBFBgM00ZtzwCexb3fp6qTc7vdiJ+P42V0X8cNWPBN6OzyOWRf9ZAmS5CADZWmPikPXGLIiU9OdDMVmDuPC+7NufVb9OjMc38QXd+z1qHjYtP3+ptrDfv8WH7eCpe0ISCfdVbEQQWjCNuS7c57CHVkCMbJst3U2j+FHG1XCaPfVkF8GcpeV13LflHjVw1LrBeSOCsC3Sub93ZLaF02u3t8/Hgm8VuvCheEDGE791VdvvmlXmYHuGg7fwfVvFAeJf6Poz/xgbZ4u8TOO7vYQM9rZJTCE6tBEBYF9rzHi7AHIp5p17B53XhW8EU8vwjb1LeJde5q39AbRtCn1vi0SWK+K0FkFwyjQe8B5hyxUK/VNQDCZD2zgdTXvtCXbboUrRSrfzXmtUrUUQtDKNmUYXh2wsClqe7zivE+f9Td3gPJLg/HChr35qlPQE+Vm8vKkGEGJ8RzcjWwmAwwba7YiEwjn0RgShAtOYSQYVyMO66+x93t7xMlpEHsEHqhyZRs3PgfQp1UvooGRx6Hxri2ThIgBZao1TumH/dCiZmxVYg+AMGWu8EUtYB6tT9Yjr0CFGDMBwEQD7tcapbAKNAl2YRjCyDNPhvKs/+5VWeMSzWjXbGxGAC7XGTIKR86sNVS+9QHGF6tpJbx+tcRfIvk2zFNt2q/BzI4Jg255R4eeol2zscUsnKm2sck7HlK1NeGD1OCHy7l9KgRoS2xEX41wEASrTmHyJvkouzSEfYsbEO4Wd+EWwrQzAA18WKy1oG+ouCa+659Np7TW5CAJvpnEv9pooPfb2VrYh3BOBYm1VvA8VpoQtvPfGLKXRzna3ShkbApt4pphsw0UANtQapzAlcMW94+W93m15tYsjCD6QceQoZSfd3VcP5Jsdt+6EOiIkbiSu87kIwiimcao76dPotQMjgxWeoHhMxr92z9xv1kV+mwhmfRIXi2SZZ/G2WbWyF56n53Q5UMbFJnchXZ1XqXrVemF9ecetjjSHrb9ImuxzKW9Ck9iReHbMRQDe1Boz401/v+qepjU+bvt2y6ddvuzCP4LlG/xbM7xxRvHKSs3IqE+sgATm9vVJG5EyOxqNmWZWTCdSKvG1eoAUxSOqrHmVXKTUgHRV2bbdZnCBbODp2jU2LhL6cstlXizalTCv49tlsjIevclvbpM0/ZIYL8p4uTlIfpp+TEu641wsrtLYePQ2v0naH3mZfqSTbW48eka/P8kWCR2e0lW8+ZiMMbZF7NASGMNFAMbUGrPpNKngY90N/h5/7/HzAf9K/NrFEgwvHIcmw9NgExjIcvoNr1I6BEEoIjYW34hgFJl+Y/HhCYJOhC6P7iEHUrNwKTea+Y5OdPRV+2dIPy8S+pKLa+PR82W8uqSP9TvjG7q8rabOK/rOVV442zvccSEpbvN8eberaAovk+UHuj4gxms6N7NWdS+KZPsD1cqWFcE+OqPPmFwl2WXlCLXNCTVydd+BIzKlFkGYwjRmwhQlX+vuOxT9vuvvvX4+7N9O3vT3how34YETJP1QssaR30hN96veG8SzAoEnXATgSa0xkwQOBd8OfkiImOWEYv2GF/Zx81373ztx+B5ZWTKslCwR8YIWWWoRhCxM49QLH+MWiO5LXSVu7uQQglNwSCQ5oVcNhimGt5D70PrEjYjpijsULgJQpNa4l2EvlMN7iXP1u9OimF6VB5sVYUf4XG978jYvy2ogujEePb5Z1f9L15Svk+rvCLj/vkhXZRpnKypka9NPdzPOU/ow2VV1EEVHjyS7iZvMr7ryf1ssY4Mf0CWrL7CBiwBsqDVmsgtRdKom8OUO7nJsh0NbjuxiAoLZVZnQ6yLU0e/2gmWsBNXlaI6zHdfdiJRJ0GjMdEoY4c5U7SWUNFMFxfYyJoSHwn2sPW4VgrjmwQ1mtq5AHSRP5SrO0piw710kW+uhMi/ogrH9i7dX+UV1vbIoffX5Os/yHfG+ILKMKbZFTM8WmMJFAKbUGjPZjPf0t27eieD7HZ+Lvm75eI9vu3iD4ImGN87oN7ccuktr760u+CVGAXFdYlqOQAouApCi1jjd3lIpdTh2x6G9yohQzC1DftQ5guwMHbKcnJ6zSFW9kpSseqXKy9kskTonlzdpESdrOnY8pu7NVl/2nE0UjiSl3ApIEEQit2oRhFtMY2YpXAPCR7fljQglEUL7oKM2OR2ASCd7+ztUdd6qNmjJZVVK/OznvLjAOKM8tLX/YXlR7fpE4WNqsvXuZ7+N15fiL2Rs8gLi+OLyjYsAbKo15rPbV3Kf7oFKhysFF7Zct+OyLnQjGPiI6H6SFJe7EP42oWPH6vOyVMT2GR0N4uwyKZSx7ZvEsUwB21wEwHatccJ29ZPoyB0Hthy347AuZCOYVxXZvSJWWjEqjQorGaYDk1ihGJjiIgCma417UUo1VCBKvWyqC9UIBm5Q7fZq4jHyyZxaQS3yAV5AIpvYkS+QgosApKg1ZhatDadzgHfY9/rneyieUWXT44sipeBfVNX5RsXxeo8iWLNvEDcvPqzb++Ufi7Qa5oxH75KMvn57v/zn+CKm0+jTIqffdpdptT4v8o9Zen33GylTXBJZ4nKfiyBMYRpzacSt6VHt8sAt7+7xqujNXS92cqC/zWUcsEx4fKHLcluhBq2iD72IU/dhR3dQ61WyWqUZfd6zdJEuP7fZtnlqtf7J1DnVBTOuyKpaBGEV05hbNe6AmNEtIlELO4l4EXGyBx9tXHTytL8XlXkqibkP1jelmq+rka3+5cv1IlU5TFFZ9Q2WNBwSy6WLiO08yI1ImaONxsQ5qocP5J4sHRjpwMahVSFGojGKx2R8dHXacEEz7w/D22Y3wkb2du3unUwZ4BuViSPcHblp1i6wcCzVIMurkPXb3/z1/wG2LBtR1gsCAA=='
API = 'H4sIAMvDwGoC/8VaW4vbRhR+D+Q/DH4ILaQQ27rYfdu4myWlXZYstJRSxFiatWdXlsxItrOE/PeMJFuWbY3O3OS+LIusM+fTuX7nSN/evkEDmjylg1/RN/4/GuRpjuMHvCAZvzR23x8v3mwxjfE8JjO2oVn5+9SZ8N+/FzcN1iyNNmFeXP63ECqPqy9/jvj1wTjIaLLA65SRwfvq9wSvSPnTL/d0scxRdTh6YukKPZ7fnHEE/Fqhw3OKS6XqVlWOWJWjqGo8gVTFOFm84B2Fdf6xv1Na+cjvVj4O5vgZBxlOgoiSRSqy60d+16lSnKDfmhLNJ4YeuFuno6NzOFF70GBFGHvN6YpYeWQH1r7EK8q1P6Us5x7fRIRFOO7QXt1/guATl+UxcC7bNIOa6cVW0HKCp2sFGW9cxR5LHMfpjpDEij1cxZgUateKSbDurHFIn2gYhCnO8gaMLU7CdLMlTGSEh0oQzQrBdjyIJBHHgWiC/jo/TqVUnGKskcHVQ4yxxtPA2GXHEYRxR7KcsCQIMaPzOeHIFjje8otpAifY35Uwmh2ET7DeHQ5Sjy6X6/5KQ46G0S0lTCrQ3D2qPytR9KUS7aEOrosKwFVg7ggcK6X9Q5H2s3NR+TrkiOuQRP73U4c6Ysh+5AzH2oVaokT1VahVW4ehyxxdRLZDx9PPc6FtTPN8pJnnmo0ezPixLh69eAbxDAEDeTpO8wydJp9jskZyjIw0ggJbq7G6xuVxqu87oaW8fhurI9tYzTw2HhpxIcy2JLbfzsa6ca3XOuDkh+K6e/51exhAXenBT0u7SSbbz9+hdsO8cpt0NdLF7X9ukJ1KtUJFIjkUm5BrlKxTTTx2UYw92UgByKZceJizTv8iiYT+8a+fRNJTiy1zge2ZYAhjkXDPGyzkWLdYAeLZauP3xrnyuejJF0qvXx87tXUk6U23sQ4maVipD7IqyzbkgtB06+Gq0yC3Xxp0mRNnwGqngwi7/X0K9eB1/cRog15jld5PuLqxarxNcaV3F67Z7mKki0MzJi1YBkrjnpLXNS4v6whn8uCspLB/9J+Qzvtd/jKY6i4IiG3aYRwppqyjD+8Eq5Yo0XbRCAIAsh4RFCWyo4NMY/3We0CJ+vOafsXMbm/2IebX0eC666HXGwMD2apsU/N6bSGeetR7lqLeMUYmnEI8kylEn2mNgzmJF3SzCrJ0k3N3rVtaTL2qr249DbRLMRXtCX+gZWExsmHpmsiguN+LoNtSxB4aP8Axzl64g4Qv6A8l6qa6sf21d5vmqTZR01vLwLEMtn/99utfZd7yTfYSvq01jqkVe7Kddi+MIoNGiHY0X6IReszTdYZw8UuW4x1+5f+8qtdT32Dm969EAcEZGwxBldFaa9tM2GoT4SAhu+A1ZS9HogGXlkr0BNA92aF/+DFWNmCw/XqyGsgxwFbg6bYCY04PB70pzwHfQmZrTJOuZn3YEz4WNyq0aPOJxzThLNR8sCYYOtDi4HpuvCiCwckQjkMnyCQ7AYi4jLX2gUmE83EvAoxIOtVBtat7V+pGHVaC/dqjveTZhtcf29Ceb+EvhvqadLXy1NPIU0s2HBYfFHMcuPRx3Nj6CPeXw+YrpXm6SSL0UB5ROjoGv342+FBhWH1OUyo9xX3c5wnfyA2b39UAuGuMkl9Eay0etcq20Wceha32xJIkixhzK9YkE+SWs1IYvSsJ5W0l3ko0dZCB/MTV5CeA5smeN4pp46Qf2ujVNL/hlG5HeGck/93BJXacIDGe/09D+UR9gTi5DnVsGdaMZjR1gn2kNvuF3ByzkMRpgruy6ZzetGzmPh7OaZZAs23dikSUq2M4KTxYAxW/Vmjc3o5NqxQzmtNsGdAsJlmXkeo6XAmgz4WAzdJziB3JWG4LHAvDj8JG1+97ozs5i5EFI+Rl76mQbmnOiWQYLikW2ag1ZkrKdFcctfdhCXTWPA/99CVdkZ/VEU87EB9TsRP7tBX7BWgoOe08TxkPO7IoOexzyqJMMiAqGfSplLEZEfrD+eQqwzmnEBGjOKdhkBFcBkBI5KL1Zi+IHglGBZ26K4VP4Nlx6/CDbGaJ6/Lwg3qgtqFvxKxBIedP1DI8d0/19RMAk7NpSDgXtED4Pb0aGaj08j//vX3z/Qciur2cbUAAAA=='
open('raw.json', 'wb').write(gzip.decompress(base64.b64decode(RAW)))
open('raw_api.json', 'wb').write(gzip.decompress(base64.b64decode(API)))
print('raw.json and raw_api.json written')

## 4. Try the live scrape (expected to be blocked in Colab)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'disney_cruise_scraper.py'], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
LIVE_OK = r.returncode == 0
print('LIVE SCRAPE', 'OK' if LIVE_OK else 'BLOCKED -> using raw.json from section 3')

## 5. Clean the data, write `results.csv`, print the answers

In [ ]:
!python disney_cruise_scraper.py --clean

## 6. Preview and download

In [ ]:
import pandas as pd
df = pd.read_csv('results.csv', encoding='utf-8-sig', keep_default_na=False)
print(df.shape, '| empty cells:', int((df == '').sum().sum()), '| duplicate rows:', int(df.duplicated().sum()))
df.drop(columns='link').head(10)

In [ ]:
from google.colab import files
files.download('results.csv')